# Step 19.3 — Training stability：先检查梯度，再决定是否更新

在仓库根目录同步母版：

```bash
git pull
```

再建立带时间戳的独立工作副本，不覆盖已有笔记：

```bash
cp -n notebooks/84_step19_3_training_stability.ipynb notebooks/work/84_step19_3_training_stability_$(date +%Y%m%d_%H%M%S)_work.ipynb
```

以上是未来使用的命令参考，本次未执行。环境仍为项目Python 3.12 / `.venv`、固定 **MLX v0.32.2**。所有代码单元未执行、输出为空；没有读取语料、创建模型或得到训练结果。

2026-11-17 教学修订：**19.3累计2/2＝历史tree工具接口修正1次＋本次教学整理1次**。保留原候选与稳定性算法，只去重旧模型、补齐讲解并将实际操作收进主动入口；其他小节及其计数不变。


## 阅读路线：本节增加的是什么？

[19.2](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dc7bbf88f36b6e1ae404ddddf0d0654540bf1894/notebooks/83_step19_2_train_validation_loop.ipynb)已经区分“更新前loss”和“更新后的参数”，并把验证与训练分开。本节在**Adam更新之前**插入一个观察与拒绝边界：先取得gradient tree，算全局范数，拒绝非有限值，再按明确选择决定是否裁剪。它不证明训练一定稳定。

| 区域 | 内容 | 使用方式 |
|---|---|---|
| 正文 | 梯度树→全局L2范数→统一缩放；更新前检查时间线 | 先读纸算与shape，再读短函数 |
| 附录A/B | 数据、sampler、完整MiniGPT、loss、验证helper | 已学依赖，只留一套；可先按链接回顾 |
| 附录C | 一组匹配的model/Adam/求导函数与单步入口 | 未来主动调用；不会在定义后自动运行 |

Notebook自包含，正文定义的函数可先出现，调用时才需要后面的模型／数据依赖。未来真正使用必须先按顺序完成所有定义；本次不执行它们，也不借旧kernel中的变量。


## 1. 初始化是起点，不是稳定性的证明

候选沿用 [19.1](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dc7bbf88f36b6e1ae404ddddf0d0654540bf1894/notebooks/82_step19_1_hyperparameter_baseline.ipynb)：C=64、N=2、H=4、T=64、B=4、学习率3e-4。模型保留原Embedding、Linear等模块的初始化，没有换初始化策略；构造前设置MLX seed。固定seed有助于在相同环境中复现随机流程，不保证跨版本、设备逐bit一致，更不表示该候选已经训练成功。

Adam继续显式使用betas=[0.9,0.999]、eps=1e-8、bias_correction=True；这是课程选择，固定版本的bias_correction默认值仍为False。梯度裁剪阈值没有被偷偷加入候选：单步默认max_grad_norm=None，表示不裁剪。

下面只是Python配置。真实vocab_size仍须由对应字符表取得；数据准备和模型构造在附录的主动入口里。


In [ ]:
baseline_candidate = {'model_width':64,'layers':2,'heads':4,'block_size':64,'batch_size':4,'learning_rate':3e-4}
seeds = {'model':2028,'train':2026,'validation':2027}


## 2. Gradient tree不是一个大数组

`nn.value_and_grad(model, loss_fn)`返回标量loss与一棵梯度树。嵌套dict/list的路径对应模型的可训练参数，每个叶子是梯度数组，shape与对应参数相同。例如一个权重叶子可为(out,in)，bias叶子可为(out,)；整个模型的叶子不必同shape。

想测量“整体有多大”，数学上可把所有叶子的所有元素视为同一长向量；实现不必真的concatenate。`tree_flatten`来自`mlx.utils`，返回`(路径, 叶子)`列表，**只展开容器，不把每片数组展成一维**。列表推导里的`for _, leaf`忽略路径、保留数组本身。

设第j片叶子为g⁽ʲ⁾，全局L2范数（global L2 norm）为：

$$
s_j=\sum_i (g_i^{(j)})^2,\qquad
G=\sqrt{\sum_j s_j}.
$$

| 步骤 | 输入shape | 输出shape |
|---|---|---|
| `mx.square(leaf)` | 本片参数的任意shape | 同shape，逐元素平方 |
| `mx.sum(...)`（不指定axis） | 同上 | 标量()：本片平方和 |
| Python `sum(squared_sums)` | 若干MLX标量组成的列表 | 标量()：各片平方和之和 |
| `mx.sqrt(...)` | 标量() | 标量()：整棵树的G |

Python的sum在这里组合MLX数组运算，不等于已取得Python数值；求值边界随后再指定。G不是loss，也不是Adam更新后的参数移动距离：Adam还使用m/v、学习率和epsilon。


### 先手算两片叶子，再看短函数

设一个虚构梯度树只有A=[3,4]（shape (2,)）和B=[12]（shape (1,)）：

| 叶子 | 逐元素平方 | 本片平方和 | 本片局部L2范数 |
|---|---|---:|---:|
| A | [9,16] | 25 | 5 |
| B | [144] | 144 | 12 |

全局是sqrt(25+144)=13，**不是5+12，也不是两片范数的平均值**。这只是纸面算例，不是MiniGPT的梯度或某次输出。

下面保留原实现。输入契约是本课程实数浮点参数对应的非空梯度树，叶子都是MLX数组；它不接收掺杂配置字符串的任意对象树，也不处理稀疏／复数梯度。空树会报错。没有改为低精度训练或新增累加策略。


In [ ]:
import mlx.core as mx
from mlx.utils import tree_flatten, tree_map

def global_grad_norm(grad_tree):
    leaves = tree_flatten(grad_tree)
    squared_sums = [mx.sum(mx.square(leaf)) for _, leaf in leaves]
    if not squared_sums:
        raise ValueError('Gradient tree has no array leaves')
    return mx.sqrt(sum(squared_sums))


## 3. Global norm clipping：每片乘同一个数

选择有限正阈值c，定义一个标量缩放系数α，再将它广播到每片叶子的每个元素：

$$
\alpha=\min\left(1,\frac{c}{G+\epsilon_{clip}}\right),\qquad
\widetilde g^{(j)}=\alpha g^{(j)}.
$$

`epsilon_clip=1e-12`用于分母，和Adam的eps是不同位置的量。在实数算术下，缩放后的全局范数为αG，不超过c；浮点计算仍有舍入误差。由于分母含epsilon，G在阈值附近可能被微幅缩小，不能把代码说成严格的“G≤c永不改变”。零梯度缩放后仍是零。

沿用A=[3,4]、B=[12]、G=13，若c=6.5，忽略这个很小的epsilon作纸算，α≈0.5，得到A≈[1.5,2]、B≈[6]，整树范数≈6.5。各分量比例统一保留；不是对不同层分别选择scale。

### 与两种容易混淆的操作比较

| 操作 | 同一示例的结果／边界 |
|---|---|
| 全局范数裁剪 | 所有叶子共用α≈0.5，整树约6.5 |
| 每片局部范数各自限到6.5 | A的范数5不变，B缩到6.5；整树sqrt(25+42.25)≈8.20，仍大于6.5 |
| 逐元素裁剪（elementwise clipping） | 单独把每个坐标截到[-c,c]，通常改变方向；例如[3,4]按坐标截到[-1,1]得到[1,1]，而全局限到1得到约[0.6,0.8] |

裁剪发生在Adam之前，Adam接收缩放后的梯度；它不直接限制Adam参数更新的范数。裁剪也不修复坏数据、NaN或错误目标，不保证loss下降。


`tree_map(lambda g: g * scale, grad_tree)`对每片叶子应用同一标量，返回保留原dict/list结构的新树；`lambda`是没有单独名字的小函数。它不会原地把原梯度数组改写，也不是再次展平／重建成一个大向量。

下面函数返回三项：新梯度树、裁剪前norm、scale。norm和scale都为标量()。它先求值norm以形成主机侧观察边界，乘法产生的新叶子仍可能是惰性数组，后面的更新／求值会使用它们。

保留原helper的范围：调用者必须传入可用float32表达的有限正max_norm，并保留正常正eps_clip。现有参数检查只拒绝非int/float或≤0，没有完整拒绝NaN/Inf等错误阈值；不要把它当成通用参数校验器。单步的None分支直接跳过本helper。范数非有限时，必须在进入裁剪前停止，而不是尝试用scale=0“救回”NaN/Inf。


In [ ]:
def clip_grad_tree_global_norm(grad_tree, max_norm, eps_clip=1e-12):
    if not isinstance(max_norm, (int, float)) or max_norm <= 0:
        raise ValueError('max_norm must be positive')
    norm = global_grad_norm(grad_tree)
    mx.eval(norm)
    scale = mx.minimum(mx.array(1.0, dtype=mx.float32), mx.array(max_norm, dtype=mx.float32) / (norm + eps_clip))
    clipped = tree_map(lambda g: g * scale, grad_tree)
    return (clipped, norm, scale)


## 4. 非有限值检查，究竟检查了什么？

NaN（Not a Number）与正／负Inf都不满足isfinite。`mx.isfinite`生成逐元素布尔值，`mx.all`归约为布尔标量，`.item()`把它取回Python，才能驱动if/raise；因此这里是**主机侧检查**。名字scalar_is_finite对应本节传入的loss和norm两枚标量。

当前范数是实数梯度平方和：叶子含NaN/Inf会使G异常；但即使叶子本身有限，平方或累加也可能溢出为Inf。所以“norm非有限”不能唯一定位坏叶子或诊断根因。代码没有逐叶定位报告，也没有改用更稳定的累加算法。

`assert_finite_training_values`先查loss，再查G，任一失败即抛FloatingPointError。本单步里它在Adam.update之前；异常不会继续落到更新行。若loss先失败，函数不会再检查G。这不是MLX的自动训练保护，是这段Python代码明确安排的顺序。


In [ ]:
def scalar_is_finite(value):
    mx.eval(value)
    return bool(mx.all(mx.isfinite(value)).item())

def assert_finite_training_values(loss, grad_norm):
    if not scalar_is_finite(loss):
        raise FloatingPointError('Training loss is NaN or Inf')
    if not scalar_is_finite(grad_norm):
        raise FloatingPointError('Gradient norm is NaN or Inf')


### 检查范围：有限不等于合理，合理也不等于收敛

| 情况 | 本节实际行为 | Adam是否可以继续？ |
|---|---|---|
| loss为NaN/Inf | 更新前抛异常 | 不调用本步Adam.update |
| loss有限、G为NaN/Inf | 更新前抛异常 | 不调用本步Adam.update |
| loss很大但有限 | **没有loss大小阈值**，有限值检查通过 | 原实现仍可更新；不能把它描述成已自动阻止发散 |
| G很大但有限，max_grad_norm=None | 不裁剪 | 使用原梯度更新 |
| G有限，明确给了合格正阈值 | 统一缩放后更新 | 不是“超阈值报错”；阈值控制缩放 |

原实现没有连续趋势判断、loss_threshold、更新后的参数/m/v有限值检查或回滚。成功走过更新前检查，不能保证Adam的新状态仍有限。若以后实际观察到异常，应停止后续更新并调查输入／目标、学习率与数值范围；本节不虚构自动诊断或恢复机制。

较大的单次loss不是发散的证明；判断趋势需要多次真实、可比较的观测，而本母版没有执行任何一次。


## 5. 未执行的一步时间线

沿用19.2：更新前参数为θ[k−1]，该批数据为Dₖ。loss与梯度都从这版参数计算：

$$
L_k=L(\theta_{k-1};D_k),\quad g_k=\nabla_\theta L(\theta_{k-1};D_k),\quad G_k=\|g_k\|_2.
$$

| 顺序 | 原stability_aware_step里的动作 | 参数／状态意义 |
|---:|---|---|
| 1 | loss_and_grad_fn(model,X,Y) | 创建旧参数下的loss和梯度计算；尚未调用Adam |
| 2 | global_grad_norm(grads) | 从原梯度树构造裁剪前G |
| 3 | mx.eval(loss_before_update,grad_norm) | 完成检查所需数组及依赖 |
| 4 | assert_finite_training_values(loss,G) | 先loss后G；失败直接终止，下面的Adam不能执行 |
| 5（可选） | clip_grad_tree_global_norm；求值preclip_norm、scale | 合格输入下统一缩放；helper复算的仍是裁剪前norm |
| 6 | optimizer.update(model,grads) | 仅此处推进Adam与模型；裁剪关闭时用原梯度 |
| 7 | mx.eval(training_state) | 完成当前模型与Adam状态；**不是更新后有限性检查** |
| 8 | 返回(loss_before_update,grad_norm) | 两项都是旧参数下的观察；G是裁剪前值，不是裁剪后范数 |

这条时间线是对源码的阅读，不是运行日志。返回值不会因为Adam已经更新而自动改成新参数的loss。没有调用验证，也没有重新forward计算更新后loss。

主机if、`.item()`、显式求值和异常拒绝保留在这个**未编译（uncompiled）单步**中；不能把整个函数直接套进19.2的mx.compile，假设行为仍然正确。附录C给出与同一model/Adam绑定的完整实现。


## 6. 与19.2循环连接时，先核对返回契约

19.2的train_step只返回一个loss；这里返回 `(loss_before_update, grad_norm)`。不能直接把整对tuple交给旧循环，再对tuple调用`.item()`。若未来只需沿用旧history字段，可以在同一setup上写一个小adapter，调用本单步后只返回loss；若要记录G，需明确增加记录，而不是把G塞进loss字段。本节的注释示例先保留两项并解包，不改变19.2文件。

训练模式由匹配setup设为train；未来每次训练调用前明确model.train()。验证helper直接算loss、不接optimizer，并在finally恢复原模式。model.eval()是模式开关，不是关闭梯度或数组求值；mx.eval是求值。当前模型没有Dropout/BatchNorm或依赖training标志的前向分支。

自查（不附答案）：

1. 给另一棵两叶梯度树，按“平方和→求和→开方”的顺序写出全局范数。
2. 为什么分别限制每片范数，不保证整树范数满足同一阈值？
3. 本函数返回G很大但有限，是否表示Adam未更新？需要同时看哪个参数？

下面是已学前置和完整调用边界。它们为独立工作副本提供依赖，不是要求跳过正文直接Run All训练。


## 附录A · 旧数据依赖与主动准备

配置映射沿用[19.1](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dc7bbf88f36b6e1ae404ddddf0d0654540bf1894/notebooks/82_step19_1_hyperparameter_baseline.ipynb)，缓存／顺序划分／采样分别来自[16.1](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dc7bbf88f36b6e1ae404ddddf0d0654540bf1894/notebooks/67_step16_1_acquire_inspect_tiny_shakespeare.ipynb)、[16.2](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dc7bbf88f36b6e1ae404ddddf0d0654540bf1894/notebooks/68_step16_2_train_validation_split.ipynb)、[16.3](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dc7bbf88f36b6e1ae404ddddf0d0654540bf1894/notebooks/69_step16_3_batch_sampler.ipynb)。保留原接口，先分90/10，再在各split内部取T+1，不跨边界制造训练目标。获取默认关闭；缓存缺失或任一split短于T+1时停止，不用toy文本替代真实语料。

以下均为导入和定义。定义下载或读取函数不访问语料，只有以后主动调用才有相应作用。


In [ ]:
import math
import random
from pathlib import Path
from urllib.request import urlopen
import mlx.nn as nn
import mlx.optimizers as optim


In [ ]:
def make_model_kwargs(c, vocab_size):
    return {'vocab_size': vocab_size, 'dims': c['model_width'], 'num_heads': c['heads'], 'num_blocks': c['layers'], 'max_seq_len': c['block_size']}

def make_optimizer_kwargs(c):
    return {'learning_rate': c['learning_rate'], 'betas': [0.9, 0.999], 'eps': 1e-08, 'bias_correction': True}


In [ ]:
SOURCE_URL='https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'

def find_project_root(start):
    current = Path(start).resolve()
    for p in [current] + list(current.parents):
        if (p / 'pyproject.toml').is_file() and (p / 'notebooks' / '82_step19_1_hyperparameter_baseline.ipynb').is_file():
            return p
    raise FileNotFoundError('Set PROJECT_ROOT to repository root.')

def acquire_tiny_shakespeare(path, source_url=SOURCE_URL):
    path = Path(path)
    if path.is_file():
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    with urlopen(source_url, timeout=30) as response:
        raw = response.read()
    tmp = path.with_suffix('.tmp')
    tmp.write_bytes(raw)
    tmp.replace(path)
    return path


loader返回真实字符表、stoi/itos和一维token stream；split按同一stream顺序切分。保留词表大小与模型构造的连接，不能用任意V掩盖数据未准备好。以下函数只定义，不在单元末尾调用。


In [ ]:
def load_cached_character_corpus(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Cache missing: {path.resolve()}; correct path or explicitly acquire it.')
    text = path.read_text(encoding='utf-8')
    chars = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for i, ch in enumerate(chars)}
    return (text, chars, stoi, itos, mx.array([stoi[ch] for ch in text], dtype=mx.int32))

def split_train_validation(stream, fraction=0.9):
    if len(stream.shape) != 1 or stream.shape[0] < 2 or (not 0 < fraction < 1):
        raise ValueError('Invalid stream/split')
    i = int(stream.shape[0] * fraction)
    if i <= 0 or i >= stream.shape[0]:
        raise ValueError('Empty split')
    return (stream[:i], stream[i:], i)


Sampler保持原版：输入选定split和持久RNG，先抽起点，再取T+1项。X/Y均为(B,T)，Y已经右移一位；loss不再shift。训练RNG持续向前，验证使用自己的局部RNG。


In [ ]:
def get_batch(data, batch_size: int, block_size: int, rng: random.Random):
    """Sample shifted (X, Y) from one already-selected integer token split."""
    if len(data.shape) != 1:
        raise ValueError('data must be one token stream with shape (N_split,)')
    if type(batch_size) is not int or batch_size < 1:
        raise ValueError('batch_size must be a positive Python int')
    if type(block_size) is not int or block_size < 1:
        raise ValueError('block_size must be a positive Python int')

    N_split = data.shape[0]
    if N_split < block_size + 1:
        raise ValueError('the selected split needs at least block_size + 1 tokens')

    starts = [
        rng.randrange(0, N_split - block_size)
        for _ in range(batch_size)
    ]

    xs = []
    ys = []
    for start in starts:
        chunk = data[start : start + block_size + 1]  # (T+1,)
        xs.append(chunk[:-1])                         # (T,)
        ys.append(chunk[1:])                          # (T,)

    X = mx.stack(xs, axis=0)  # (B,T)
    Y = mx.stack(ys, axis=0)  # (B,T)
    return X, Y, starts


### 明确的数据读取入口

把原顶层bootstrap收进prepare_cached_data。它显式接收project_root和config，默认acquire_missing_data=False；主动调用才会读取缓存、建立字符表、划分并求值token数组。返回原先所需字段，不往隐藏全局变量里写数据，也不采样或创建模型。


In [ ]:
def prepare_cached_data(project_root, config, *, acquire_missing_data=False):
    root = find_project_root(project_root)
    cache_path = root / "data" / "tinyshakespeare" / "input.txt"
    if acquire_missing_data:
        acquire_tiny_shakespeare(cache_path)
    text, characters, stoi, itos, token_stream = load_cached_character_corpus(cache_path)
    train_tokens, val_tokens, split_index = split_train_validation(token_stream, 0.90)
    vocab_size = len(characters)
    for name, split in (("train", train_tokens), ("validation", val_tokens)):
        if split.shape[0] < config["block_size"] + 1:
            raise ValueError(f"{name} split needs block_size+1 tokens")
    mx.eval(train_tokens, val_tokens)
    return {
        "PROJECT_ROOT": root, "CACHE_PATH": cache_path,
        "text": text, "characters": characters, "stoi": stoi, "itos": itos,
        "token_stream": token_stream, "vocab_size": vocab_size,
        "train_tokens": train_tokens, "val_tokens": val_tokens, "split_index": split_index,
    }


## 附录B · 已学模型、loss与验证，只留一套

原403行模型块中CausalMultiHeadAttention、FeedForwardMLP、TransformerBlock各写了三次；最后生效版本彼此等价，并与[15.4完整MiniGPT](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dc7bbf88f36b6e1ae404ddddf0d0654540bf1894/notebooks/65_step15_4_implement_minigpt.ipynb)及已验收19.2/19.4一致。这里每类只保留一次，没有改变实际层数、bias配置或参数结构。

以下长代码是**可选前置恢复**，不是本节的新梯度算法。MHA的输入／输出为(B,T,C)，score为(B,H,T,T)，合并QKV与拆头细节见[13.3](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dc7bbf88f36b6e1ae404ddddf0d0654540bf1894/notebooks/54_step13_3_split_channels_into_heads.ipynb)。类定义不创建实例。


In [ ]:
class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)


### 旧MLP与Block

[14.4](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dc7bbf88f36b6e1ae404ddddf0d0654540bf1894/notebooks/60_step14_4_feed_forward_mlp.ipynb)的带bias两层MLP仍为C→4C→C，GELU不变。Block保留两条Pre-Norm RMSNorm残差路径，shape仍为(B,T,C)。这里没有引入新的激活函数或归一化策略。


In [ ]:
class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))

class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


### 完整MiniGPT汇总定义

token IDs (B,T)经过token/learned-position embeddings、N个独立Block、final RMSNorm与untied head，得到raw logits (B,T,V)。V匹配真实字符表，T不能超过max_seq_len。下面保留原检查与参数结构，不自动构造模型。


In [ ]:
class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dims: int,
        num_heads: int,
        num_blocks: int,
        max_seq_len: int,
    ):
        super().__init__()

        if vocab_size <= 0:
            raise ValueError('vocab_size must be positive')
        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if num_blocks <= 0:
            raise ValueError('num_blocks must be positive')
        if max_seq_len <= 0:
            raise ValueError('max_seq_len must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        # Python metadata: these are not trainable mx.array parameters.
        self.vocab_size = vocab_size
        self.dims = dims
        self.max_seq_len = max_seq_len

        # Learned lookup tables.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            dims=dims,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            dims=dims,
        )

        # A list of distinct Module instances; MLX recursively tracks them.
        self.blocks = [
            TransformerBlock(
                dims=dims,
                num_heads=num_heads,
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = nn.RMSNorm(dims=dims)
        self.lm_head = nn.Linear(
            input_dims=dims,
            output_dims=vocab_size,
            bias=False,
        )

    def __call__(self, token_ids: mx.array) -> mx.array:
        if len(token_ids.shape) != 2:
            raise ValueError('token_ids must have shape (B,T)')

        B, T = token_ids.shape

        if T < 1:
            raise ValueError('sequence length T must be at least 1')
        if T > self.max_seq_len:
            raise ValueError(
                f'sequence length {T} exceeds max_seq_len {self.max_seq_len}'
            )

        # Forward-local structural indices: not Module parameters.
        position_ids = mx.arange(T)  # (T,)

        token_features = self.token_embedding(token_ids)        # (B,T,C)
        position_features = self.position_embedding(position_ids)  # (T,C)
        x = token_features + position_features                  # (B,T,C)

        for block in self.blocks:
            x = block(x)  # invariant: (B,T,C)

        x = self.final_norm(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,V)

        expected_shape = (B, T, self.vocab_size)
        if logits.shape != expected_shape:
            raise RuntimeError(
                f'logits shape {logits.shape} != expected {expected_shape}'
            )

        return logits


### 复用scalar loss与validation helper

[17.2](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dc7bbf88f36b6e1ae404ddddf0d0654540bf1894/notebooks/73_step17_2_flatten_bt_supervision.ipynb)的loss先把(B,T,V)与(B,T)按相同顺序展平，cross-entropy后取mean，返回标量()。传入的是raw logits，不预先softmax，也不再次shift Y。


In [ ]:
def loss_fn(model, X, Y):
    # X, Y: (B, T); Y is already shifted by get_batch.
    logits = model(X)  # (B, T, V), raw unnormalized logits
    B, T, V = logits.shape
    flat_logits = mx.reshape(logits, (B * T, V))  # (B*T, V)
    flat_targets = mx.reshape(Y, (B * T,))        # (B*T,)
    token_losses = nn.losses.cross_entropy(
        flat_logits, flat_targets, axis=-1, reduction="none"
    )  # (B*T,)
    return mx.mean(token_losses)


验证helper沿用原实现，职责与[19.2](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/dc7bbf88f36b6e1ae404ddddf0d0654540bf1894/notebooks/83_step19_2_train_validation_loop.ipynb)一致：每次重建自己的validation RNG，参数固定时对多个batch按监督位置数加权，直接调用loss_fn而不调用optimizer；finally恢复模式。它没有新增稳定性检查或训练更新，不把两批抽样估计声称为完整验证集结果。


In [ ]:
def estimate_validation_loss(model, val_tokens, batch_size, block_size, num_batches, seed):
    if type(num_batches) is not int or num_batches < 1:
        raise ValueError('num_batches must be positive')
    rng = random.Random(seed)
    was_training = model.training
    model.eval()
    try:
        total = 0.0
        positions = 0
        for _ in range(num_batches):
            Xv, Yv, _ = get_batch(val_tokens, batch_size, block_size, rng)
            lv = loss_fn(model, Xv, Yv)
            mx.eval(lv)
            n = Yv.shape[0] * Yv.shape[1]
            total += lv.item() * n
            positions += n
        return total / positions
    finally:
        model.train(was_training)


## 附录C · 匹配setup和未编译单步

make_stability_setup把原模型构造、nn.value_and_grad、Adam.init和state容器放在一起。**只有主动调用factory时**才创建并求值初始状态；定义factory不会初始化模型。MLX seed在构造前设置；train_rng另用Python random.Random，不能混同这两个随机数来源。

内部stability_aware_step是闭包（closure）：它绑定本次创建的同一个model、optimizer、求导函数与training_state，不借旧kernel里的全局对象。它的(X,Y,max_grad_norm=None)接口与原算法保留，返回两项旧参数观察。state保存原容器引用，不能换新model却沿用旧求导函数或旧state。

逐行将下面单步对照正文的八步时间线：有限值拒绝在Adam前，更新后的mx.eval仅负责执行。这里没有mx.compile装饰器，也不新增loss阈值、回滚或自动调参。


In [ ]:
def make_stability_setup(config, vocab_size, seed_config):
    model_kwargs = make_model_kwargs(config, vocab_size)
    mx.random.seed(seed_config['model'])
    model = MiniGPT(**model_kwargs)
    model.train()
    loss_and_grad_fn = nn.value_and_grad(model, loss_fn)
    optimizer = optim.Adam(**make_optimizer_kwargs(config))
    optimizer.init(model.trainable_parameters())
    training_state = [model.state, optimizer.state]
    mx.eval(training_state)

    def stability_aware_step(X, Y, max_grad_norm=None):
        loss_before_update, grads = loss_and_grad_fn(model, X, Y)
        grad_norm = global_grad_norm(grads)
        mx.eval(loss_before_update, grad_norm)
        assert_finite_training_values(loss_before_update, grad_norm)
        if max_grad_norm is not None:
            grads, preclip_norm, scale = clip_grad_tree_global_norm(grads, max_grad_norm)
            mx.eval(preclip_norm, scale)
        optimizer.update(model, grads)
        mx.eval(training_state)
        return (loss_before_update, grad_norm)
    train_rng = random.Random(seed_config["train"])
    return {
        "model": model, "optimizer": optimizer, "training_state": training_state,
        "stability_aware_step": stability_aware_step, "train_rng": train_rng,
    }


### 可选未来入口：读数据、建状态、做一步，三个不同动作

下面全为注释，Run All不会调用任何读取、初始化、采样或更新入口。未来在工作副本明确决定操作时，先完成全部定义，再分段使用；第三段会实际采样并更新，不能当成“只是展示代码”。

保留默认max_grad_norm=None；纸算的6.5不是为真实模型推荐的阈值。训练RNG从同一个setup取用，重复采样不重置seed。初始化新setup会重建整组模型／Adam状态，不是继续旧训练；继续同一组时不要只重建其中一个对象。

本例将返回tuple解包，若未来需要19.2旧循环的loss-only接口，应显式适配；此处不复制整段循环或自动启动验证。


In [ ]:
# 1. 未来主动准备数据：默认不下载，本次不取消注释。
# PROJECT_ROOT = Path("/absolute/path/to/mlx-minigpt-from-scratch")
# selected_config = baseline_candidate
# data = prepare_cached_data(PROJECT_ROOT, selected_config,
#                            acquire_missing_data=False)

# 2. 未来主动创建一组匹配的新状态；不是训练更新。
# setup = make_stability_setup(selected_config, data["vocab_size"], seeds)

# 3. 未来主动采样和训练一步；取消注释会执行真实操作。
# setup["model"].train()
# X, Y, _ = get_batch(data["train_tokens"], selected_config["batch_size"],
#                     selected_config["block_size"], setup["train_rng"])
# loss_before_update, grad_norm = setup["stability_aware_step"](
#     X, Y, max_grad_norm=None,
# )
# mx.eval(loss_before_update, grad_norm)
# print({"train_loss_before_update": loss_before_update.item(),
#        "gradient_norm_before_clip": grad_norm.item()})


## 固定版本依据与交付状态

- [MLX v0.32.2 utils.py](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/utils.py)：tree_flatten返回路径/叶子，tree_map保留容器结构；两者属于mlx.utils。
- [v0.32.2 nn.value_and_grad](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/nn/utils.py)：对同一model的可训练参数求梯度。
- [v0.32.2 Optimizer/Adam](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/optimizers/optimizers.py)：update、状态、step与显式bias_correction。

本次只改19.3：相同模型定义去重，短代码前补树结构、shape、纸算、检查范围和一步时间线；旧数据与初始化顶层动作移为明确入口。模型、sampler、loss、范数／裁剪／finite helpers、validation与单步稳定性逻辑均按原版静态对照，没有新增训练算法。

全部代码单元execution_count=null、outputs=[]。本轮只解析JSON/schema和Python语法、比较AST及依赖；没有执行Notebook、配置/测试单元、训练、验证、实验、采样、绘图、测量、语料读取下载、checkpoint或用户工作副本。没有生成真实结果或宣称运行通过／学员掌握。

累计19.3修正2/2（历史接口修正1次＋本次教学修订1次）；19.2仍2/2，其他历史不变。本轮停在19.3等待独立验收，不推进其他节。
